# Yoga Pose Detection - Video Inference

This notebook runs trained pose-classification models on a video file.
It:
- Extracts MediaPipe landmarks from every Nth frame
- Engineers the exact same features used during training (from the pose YAML config)
- Predicts the pose class + probability using a saved pipeline
- Displays the annotated video inline with skeleton overlay and prediction text

**Works for any pose** - just update the paths in Cell 1.

---
## Cell 1 - Configuration (edit this cell only)

In [16]:
from pathlib import Path

#  Pose identity 
POSE_NAME       = "mountain_pose"
EXPERIMENT_TAG  = "4class_no_bent_forward"   # matches the tag used during training

#  Saved artefact paths ─
# Root that contains models/ metadata/ results/ sub-folders
SAVED_FILES_ROOT = Path("../models/saved_files") / f"{POSE_NAME}_files"

DIR_MODELS   = SAVED_FILES_ROOT / "models"
DIR_METADATA = SAVED_FILES_ROOT / "metadata"
DIR_RESULTS  = SAVED_FILES_ROOT / "results"

#  Model selection 
# Set to None  → auto-selects the best model from the results CSV (recommended)
# Set to a string to force a specific model, e.g. "svc" or "xgboost"
FORCE_MODEL: str | None = None

AVAILABLE_MODELS = ["logistic_regression", "svc", "knn", "xgboost"]

#  YAML config (must be the same YAML used during feature engineering) 
YAML_PATH = Path("../../configs/poses/mountain_pose.yaml")

# Predictions below this confidence are shown as "unknown"
# Set to 0.0 to disable thresholding
PROBABILITY_THRESHOLD = 0.65

# Smoothing — sliding window majority vote
SMOOTHING_WINDOW  = 15   # number of recent frames to vote over
MIN_HOLD_FRAMES   = 10   # minimum frames to hold a prediction before switching

#  Feature columns (must match the order used during training) 
FEATURE_COLUMNS = [
    "left_torso_hip_angle",
    "right_torso_hip_angle",
    "left_shoulder_arm_angle",
    "right_shoulder_arm_angle",
    "neck_tilt_angle",
    "feet_distance_normalized",
    "ear_shoulder_lateral_delta",
    "plumb_line_alignment",
]

#  Video 
VIDEO_PATH = Path("test_videos/mountain_test.mp4")   # relative to this notebook
FRAME_STEP = 1    # process every Nth frame (1 = every frame, 2 = every other)

#  Playback window 
WINDOW_NAME  = f"{POSE_NAME} - Pose Detection"
# cv2.waitKey delay in ms between frames. Lower = faster playback.
# 1  → as fast as inference allows (real-time feel on most machines)
# 33 → ~30 fps cap
WAITKEY_MS   = 1
# Resize the display window to this width (height scales proportionally).
# Set to None to keep original video resolution.
DISPLAY_WIDTH: int | None = 960

#  MediaPipe model 
# Will be auto-downloaded next to this notebook if not present
MEDIAPIPE_MODEL_PATH = Path("pose_landmarker_lite.task")
MEDIAPIPE_MODEL_URL  = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
)

print("Configuration loaded.")
print(f"  Pose           : {POSE_NAME}")
print(f"  Experiment tag : {EXPERIMENT_TAG}")
print(f"  Saved files    : {SAVED_FILES_ROOT}  (exists={SAVED_FILES_ROOT.exists()})")
print(f"  YAML           : {YAML_PATH}  (exists={YAML_PATH.exists()})")
print(f"  Video          : {VIDEO_PATH}  (exists={VIDEO_PATH.exists()})")
print(f"  Force model    : {FORCE_MODEL or 'auto (best from results CSV)'}")

Configuration loaded.
  Pose           : mountain_pose
  Experiment tag : 4class_no_bent_forward
  Saved files    : ..\models\saved_files\mountain_pose_files  (exists=True)
  YAML           : ..\..\configs\poses\mountain_pose.yaml  (exists=True)
  Video          : test_videos\mountain_test.mp4  (exists=True)
  Force model    : auto (best from results CSV)


---
## Cell 2 - Imports

In [3]:
import math
import urllib.request
import warnings

import cv2
import joblib
import mediapipe as mp
import numpy as np
import pandas as pd
import yaml
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import RunningMode

print("All imports OK.")

All imports OK.


---
## Cell 3 - Skeleton & colour constants

In [4]:
# Full MediaPipe Pose skeleton connections (33 landmarks)
SKELETON_CONNECTIONS = [
    # Face
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    # Torso
    (11, 12), (11, 23), (12, 24), (23, 24),
    # Left arm
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    # Right arm
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    # Left leg
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    # Right leg
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
]

# Feature-relevant landmark pairs derived from mountain_pose.yaml
# These are highlighted in a distinct colour so the user can see which joints
# drive the classification decision.
#
# joint_angles   → draw each arm of the angle as a separate pair
# spatial_distances / alignment_offsets → draw lines between the listed joints
FEATURE_CONNECTIONS = [
    # left_torso_hip_angle  joints: [11, 23, 25]
    (11, 23), (23, 25),
    # right_torso_hip_angle  joints: [12, 24, 26]
    (12, 24), (24, 26),
    # left_shoulder_arm_angle  joints: [23, 11, 15]
    (23, 11), (11, 15),
    # right_shoulder_arm_angle  joints: [24, 12, 16]
    (24, 12), (12, 16),
    # neck_tilt_angle  joints: [7, 11, 23]
    (7, 11), (11, 23),
    # feet_distance_normalized  joints: [27, 28]  norm: [11, 12]
    (27, 28), (11, 12),
    # ear_shoulder_lateral_delta  joints: [7, 8, 11, 12]
    (7, 8), (11, 12),
    # plumb_line_alignment  joints: [11, 23, 25, 27]  norm: [11, 23]
    (11, 23), (23, 25), (25, 27),
]
# Deduplicate while preserving order
FEATURE_CONNECTIONS = list(dict.fromkeys(FEATURE_CONNECTIONS))
FEATURE_CONNECTIONS_SET = set(FEATURE_CONNECTIONS)

# BGR colours
COLOUR_SKELETON  = (200, 200, 200)   # light grey  - standard skeleton
COLOUR_FEATURE   = (0, 220, 255)     # vivid yellow-green - feature-relevant joints
COLOUR_LANDMARK  = (255, 255, 255)   # white dots for all landmarks
COLOUR_FEAT_DOT  = (0, 220, 255)     # matching dot for feature landmarks

# Landmark indices that appear in at least one feature connection
FEATURE_LANDMARK_INDICES = set(
    idx for pair in FEATURE_CONNECTIONS for idx in pair
)

print(f"Standard skeleton connections : {len(SKELETON_CONNECTIONS)}")
print(f"Feature-relevant connections  : {len(FEATURE_CONNECTIONS)}")
print(f"Feature-relevant landmark ids : {sorted(FEATURE_LANDMARK_INDICES)}")

Standard skeleton connections : 35
Feature-relevant connections  : 13
Feature-relevant landmark ids : [7, 8, 11, 12, 15, 16, 23, 24, 25, 26, 27, 28]


---
## Cell 4 - Feature engineering helpers

These functions are a direct port of the training-time feature engineering pipeline.
They operate on a **single frame** (scalar / 1-D inputs) rather than on a DataFrame
column, which is more efficient for online inference.

In [5]:
EPSILON = 1e-6


def _xyz_from_landmarks(landmarks, idx: int) -> np.ndarray:
    """
    Extract (x, y, z) for landmark index `idx` from a MediaPipe landmark list.
    Returns a 1-D array of shape (3,).
    """
    lm = landmarks[idx]
    return np.array([lm.x, lm.y, lm.z], dtype=np.float64)


def _angle_at_vertex(
    a: np.ndarray, b: np.ndarray, c: np.ndarray
) -> float:
    """
    3-D angle at vertex B formed by rays B→A and B→C, in degrees.
    Matches the vectorised training implementation exactly.
    """
    ba = a - b
    bc = c - b
    dot = np.dot(ba, bc)
    cosine = np.clip(
        dot / (np.linalg.norm(ba) * np.linalg.norm(bc) + EPSILON), -1.0, 1.0
    )
    return float(np.degrees(np.arccos(cosine)))


def _euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Euclidean distance between two 3-D points."""
    return float(np.linalg.norm(a - b))


def _point_to_line_distance(
    p: np.ndarray, a: np.ndarray, b: np.ndarray
) -> float:
    """
    Perpendicular distance from point P to the line through A and B.
    Matches the cross-product formulation used in training.
    """
    ab = b - a
    ap = p - a
    cross = np.cross(ap, ab)
    return float(np.linalg.norm(cross) / (np.linalg.norm(ab) + EPSILON))


def _compute_joint_angle_feature(
    landmarks, cfg: dict
) -> float:
    """Compute a single joint-angle feature from a landmark config dict."""
    j = cfg["joints"]
    return _angle_at_vertex(
        _xyz_from_landmarks(landmarks, j[0]),
        _xyz_from_landmarks(landmarks, j[1]),
        _xyz_from_landmarks(landmarks, j[2]),
    )


def _compute_spatial_distance_feature(
    landmarks, cfg: dict
) -> float:
    """Compute a single spatial-distance feature, with optional normalisation."""
    j = cfg["joints"]
    dist = _euclidean_distance(
        _xyz_from_landmarks(landmarks, j[0]),
        _xyz_from_landmarks(landmarks, j[1]),
    )
    if "normalization_factor" in cfg:
        nf = cfg["normalization_factor"]
        norm_dist = _euclidean_distance(
            _xyz_from_landmarks(landmarks, nf[0]),
            _xyz_from_landmarks(landmarks, nf[1]),
        )
        dist = dist / (norm_dist + EPSILON)
    return dist


def _compute_alignment_offset_feature(
    landmarks, cfg: dict
) -> float:
    """
    Compute a single alignment-offset feature.
    Dispatch logic mirrors the training implementation:
        2 joints  → absolute lateral X-delta
        3 joints  → perpendicular distance of middle joint from outer-joint line
        4 joints + 'plumb' in name → sum of deviations of j[1], j[2] from j[0]→j[3]
        4 joints  → absolute slope-delta between pair [0,1] and pair [2,3]
    """
    j    = cfg["joints"]
    name = cfg["name"]
    n    = len(j)

    try:
        if n == 2:
            lm0 = _xyz_from_landmarks(landmarks, j[0])
            lm1 = _xyz_from_landmarks(landmarks, j[1])
            val = abs(lm0[0] - lm1[0])

        elif n == 3:
            val = _point_to_line_distance(
                _xyz_from_landmarks(landmarks, j[1]),
                _xyz_from_landmarks(landmarks, j[0]),
                _xyz_from_landmarks(landmarks, j[2]),
            )

        elif n == 4 and "plumb" in name.lower():
            outer_a = _xyz_from_landmarks(landmarks, j[0])
            outer_b = _xyz_from_landmarks(landmarks, j[3])
            dev1 = _point_to_line_distance(
                _xyz_from_landmarks(landmarks, j[1]), outer_a, outer_b
            )
            dev2 = _point_to_line_distance(
                _xyz_from_landmarks(landmarks, j[2]), outer_a, outer_b
            )
            val = dev1 + dev2

        elif n == 4:
            lm0 = _xyz_from_landmarks(landmarks, j[0])
            lm1 = _xyz_from_landmarks(landmarks, j[1])
            lm2 = _xyz_from_landmarks(landmarks, j[2])
            lm3 = _xyz_from_landmarks(landmarks, j[3])
            angle_01 = np.arctan2(lm1[1] - lm0[1], lm1[0] - lm0[0])
            angle_23 = np.arctan2(lm3[1] - lm2[1], lm3[0] - lm2[0])
            # Wrap to [-pi, +pi] to avoid boundary discontinuities
            delta_rad = (angle_01 - angle_23 + math.pi) % (2 * math.pi) - math.pi
            val = math.degrees(abs(delta_rad))

        else:
            raise ValueError(
                f"Alignment offset '{name}' has {n} joints; expected 2, 3, or 4."
            )

    except Exception as exc:
        warnings.warn(f"Failed computing alignment offset '{name}': {exc}")
        return float("nan")

    if "normalization_factor" in cfg:
        nf = cfg["normalization_factor"]
        scale = _euclidean_distance(
            _xyz_from_landmarks(landmarks, nf[0]),
            _xyz_from_landmarks(landmarks, nf[1]),
        )
        val = val / (scale + EPSILON)

    return val


def extract_features_from_landmarks(
    landmarks, feat_config: dict, feature_columns: list[str]
) -> np.ndarray:
    """
    Compute all engineered features for a single frame from MediaPipe landmarks.

    Args:
        landmarks     : pose_landmarks list from a MediaPipe detection result
        feat_config   : the `engineered_features` block loaded from the pose YAML
        feature_columns: ordered list of feature names matching training order

    Returns:
        1-D numpy array of shape (n_features,) in `feature_columns` order.
        Returns None if any feature is NaN (pose not usable for inference).
    """
    feature_map: dict[str, float] = {}

    for cfg in feat_config.get("joint_angles", []):
        feature_map[cfg["name"]] = _compute_joint_angle_feature(landmarks, cfg)

    for cfg in feat_config.get("spatial_distances", []):
        feature_map[cfg["name"]] = _compute_spatial_distance_feature(landmarks, cfg)

    for cfg in feat_config.get("alignment_offsets", []):
        feature_map[cfg["name"]] = _compute_alignment_offset_feature(landmarks, cfg)

    try:
        row = np.array([feature_map[col] for col in feature_columns], dtype=np.float64)
    except KeyError as e:
        raise KeyError(
            f"Feature {e} is in FEATURE_COLUMNS but was not computed. "
            f"Check that the YAML config covers all required features."
        ) from e

    if np.any(np.isnan(row)):
        return None

    return row


print("Feature engineering helpers defined.")

Feature engineering helpers defined.


---
## Cell 5 - Drawing helpers

In [6]:
def _landmark_px(lm, frame_w: int, frame_h: int) -> tuple[int, int]:
    """Convert normalised MediaPipe coordinates to pixel coordinates."""
    return int(lm.x * frame_w), int(lm.y * frame_h)


def draw_skeleton(
    frame: np.ndarray,
    landmarks,
) -> np.ndarray:
    """
    Draw the full pose skeleton on `frame`.

    - Standard connections are drawn in light grey.
    - Feature-relevant connections are drawn in vivid cyan-yellow on top.
    - All landmark dots are white; feature-relevant ones match the feature colour.

    Args:
        frame     : BGR frame (modified in-place and returned).
        landmarks : pose_landmarks list from MediaPipe.

    Returns:
        Annotated frame.
    """
    h, w = frame.shape[:2]

    # Precompute pixel positions for all 33 landmarks
    px = [_landmark_px(lm, w, h) for lm in landmarks]

    # Draw standard skeleton connections first (background layer)
    for a, b in SKELETON_CONNECTIONS:
        if (a, b) not in FEATURE_CONNECTIONS_SET and (b, a) not in FEATURE_CONNECTIONS_SET:
            cv2.line(frame, px[a], px[b], COLOUR_SKELETON, 2, cv2.LINE_AA)

    # Draw feature-relevant connections on top (foreground layer)
    for a, b in FEATURE_CONNECTIONS:
        cv2.line(frame, px[a], px[b], COLOUR_FEATURE, 3, cv2.LINE_AA)

    # Draw landmark dots
    for i, pos in enumerate(px):
        colour = COLOUR_FEAT_DOT if i in FEATURE_LANDMARK_INDICES else COLOUR_LANDMARK
        radius = 5 if i in FEATURE_LANDMARK_INDICES else 3
        cv2.circle(frame, pos, radius, colour, -1, cv2.LINE_AA)

    return frame


def draw_prediction_overlay(
    frame: np.ndarray,
    pose_name: str,
    predicted_class: str,
    probability: float | None,
    feedback: str,
    frame_idx: int,
) -> np.ndarray:
    """
    Render a semi-transparent info panel in the top-left corner with:
        - Pose name
        - Predicted class
        - Prediction probability (if available)
        - Corrective feedback from the YAML
        - Current frame index

    Args:
        frame           : BGR frame (modified in-place and returned).
        pose_name       : e.g. 'mountain_pose'
        predicted_class : e.g. 'correct' or 'wide_stance'
        probability     : float in [0, 1] or None if model has no predict_proba
        feedback        : corrective text from the YAML feedback block
        frame_idx       : frame number shown in the corner

    Returns:
        Annotated frame.
    """
    h, w = frame.shape[:2]
    font       = cv2.FONT_HERSHEY_SIMPLEX
    pad        = 10
    line_h     = 28
    small_scale = 0.55
    large_scale = 0.75

    prob_str = f"{probability * 100:.1f}%" if probability is not None else "N/A"

    # Choose banner colour: green for correct, amber otherwise
    is_correct     = predicted_class.lower() == "correct"
    banner_colour  = (30, 160, 30) if is_correct else (0, 130, 200)

    lines = [
        (f"Pose  : {pose_name.replace('_', ' ').title()}", small_scale, (220, 220, 220)),
        (f"Class : {predicted_class.replace('_', ' ').upper()}", large_scale, (255, 255, 255)),
        (f"Prob  : {prob_str}", small_scale, (200, 220, 255)),
        (f"Frame : {frame_idx}", small_scale, (180, 180, 180)),
    ]

    # Feedback text (word-wrap at ~45 chars)
    if feedback:
        words, current = feedback.split(), ""
        wrap_lines = []
        for word in words:
            if len(current) + len(word) + 1 > 45:
                wrap_lines.append(current.strip())
                current = word + " "
            else:
                current += word + " "
        if current:
            wrap_lines.append(current.strip())
        for wl in wrap_lines:
            lines.append((wl, small_scale - 0.05, (160, 230, 160)))

    panel_h = pad * 2 + len(lines) * line_h
    panel_w = 340

    # Draw semi-transparent dark background
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (panel_w, panel_h), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)

    # Coloured left border
    cv2.rectangle(frame, (0, 0), (5, panel_h), banner_colour, -1)

    # Draw text lines
    for row_i, (text, scale, colour) in enumerate(lines):
        y = pad + row_i * line_h + line_h // 2
        cv2.putText(frame, text, (14, y), font, scale, colour, 1, cv2.LINE_AA)

    return frame


def frame_to_jpeg_b64(frame: np.ndarray, quality: int = 85) -> str:
    """Encode a BGR frame to a base-64 JPEG string for inline HTML display."""
    ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not ok:
        raise RuntimeError("cv2.imencode failed.")
    return base64.b64encode(buf.tobytes()).decode()


print("Drawing helpers defined.")

Drawing helpers defined.


---
## Cell 6 - Load artefacts

In [7]:
#  1. Load YAML config 
assert YAML_PATH.exists(), f"YAML not found: {YAML_PATH}"
with open(YAML_PATH, "r") as f:
    pose_yaml = yaml.safe_load(f)

feat_config = pose_yaml.get("features_config", {}).get("engineered_features", {})
assert feat_config, "No 'features_config.engineered_features' block found in YAML."

yaml_feedback: dict = pose_yaml.get("feedback", {})
print(f"YAML loaded. Feedback classes: {list(yaml_feedback.keys())}")

#  2. Select model 
if FORCE_MODEL is not None:
    assert FORCE_MODEL in AVAILABLE_MODELS, (
        f"FORCE_MODEL='{FORCE_MODEL}' is not in {AVAILABLE_MODELS}"
    )
    selected_model = FORCE_MODEL
    print(f"Model forced to: {selected_model}")
else:
    # Auto-select: pick the model with the highest test_f1_weighted from results CSV
    results_csv = DIR_RESULTS / f"{POSE_NAME}_{EXPERIMENT_TAG}_experiment_results.csv"
    assert results_csv.exists(), (
        f"Results CSV not found: {results_csv}\n"
        f"Set FORCE_MODEL to specify a model manually."
    )
    results_df   = pd.read_csv(results_csv)
    best_row     = results_df.sort_values("test_f1_weighted", ascending=False).iloc[0]
    selected_model = best_row["model"]
    print(f"Results CSV loaded. Best model by F1: {selected_model} "
          f"(test_f1={best_row['test_f1_weighted']:.4f})")

#  3. Load pipeline and label encoder 
prefix        = f"{POSE_NAME}_{EXPERIMENT_TAG}_{selected_model}"
pipeline_path = DIR_MODELS / f"{prefix}_pipeline.joblib"
encoder_path  = DIR_MODELS / f"{prefix}_label_encoder.joblib"

assert pipeline_path.exists(), f"Pipeline not found: {pipeline_path}"
assert encoder_path.exists(),  f"Label encoder not found: {encoder_path}"

pipeline      = joblib.load(pipeline_path)
label_encoder = joblib.load(encoder_path)

print(f"Pipeline loaded   : {pipeline_path.name}")
print(f"Label encoder     : {encoder_path.name}")
print(f"Known classes     : {list(label_encoder.classes_)}")

#  4. Check probability support 
HAS_PROBA = hasattr(pipeline, "predict_proba")
print(f"Probability support: {HAS_PROBA}")

YAML loaded. Feedback classes: ['correct', 'arms_not_dropped', 'bent_forward', 'tilted_head', 'wide_stance']
Results CSV loaded. Best model by F1: logistic_regression (test_f1=0.9484)
Pipeline loaded   : mountain_pose_4class_no_bent_forward_logistic_regression_pipeline.joblib
Label encoder     : mountain_pose_4class_no_bent_forward_logistic_regression_label_encoder.joblib
Known classes     : ['arms_not_dropped', 'correct', 'tilted_head', 'wide_stance']
Probability support: True


---
## Cell 7 - Ensure MediaPipe model file

In [8]:
def ensure_mediapipe_model(model_path: Path, url: str) -> None:
    """Download the MediaPipe .task model file if it is not already present."""
    if model_path.exists():
        print(f"MediaPipe model found: {model_path}")
        return
    print(f"Downloading MediaPipe model from:\n  {url}")
    urllib.request.urlretrieve(url, model_path)
    print(f"Saved to: {model_path}")


ensure_mediapipe_model(MEDIAPIPE_MODEL_PATH, MEDIAPIPE_MODEL_URL)
assert VIDEO_PATH.exists(), f"Video file not found: {VIDEO_PATH}"

MediaPipe model found: pose_landmarker_lite.task


---
## Cell 8 - Stream inference + display in a cv2 window

Detect → annotate → display in one tight loop - no frame buffering.
A `cv2` popup window opens automatically.

**Controls while the window is open:**
- Press **Q** or **Esc** to quit early
- Press **Space** to pause / resume

The kernel cell completes when the video ends or you quit.

**How to Tune It(in `CELL 1`)**

| Scenario | Adjustment |
|---|---|
| Still flickering | Increase `SMOOTHING_WINDOW` (try `20–30`) |
| Too slow to react to a real pose change | Decrease `MIN_HOLD_FRAMES` (try `5–8`) |
| `"unknown"` flashing briefly during transitions | Increase `MIN_HOLD_FRAMES` so it has to win convincingly |
| Testing a static video | `SMOOTHING_WINDOW=10`, `MIN_HOLD_FRAMES=8` is a good baseline |

In [14]:
from collections import deque

class PredictionSmoother:
    """
    Majority-vote smoother over a sliding window of recent predictions.
    A new class only takes over when it wins the vote AND the current
    class has been held for at least min_hold_frames.
    """
    def __init__(self, window_size: int, min_hold_frames: int):
        self.window        = deque(maxlen=window_size)
        self.min_hold      = min_hold_frames
        self.current_class = None
        self.hold_count    = 0

    def update(self, new_class: str) -> str:
        self.window.append(new_class)

        # Majority vote across the window
        vote_winner = max(set(self.window), key=self.window.count)

        if self.current_class is None:
            # First prediction — accept immediately
            self.current_class = vote_winner
            self.hold_count    = 1
        elif vote_winner == self.current_class:
            # Same class — keep counting hold time
            self.hold_count += 1
        elif self.hold_count >= self.min_hold:
            # Different class won AND we've held long enough — switch
            self.current_class = vote_winner
            self.hold_count    = 1
        else:
            # Different class won but hold time not met — stay put
            self.hold_count += 1

        return self.current_class

In [17]:
def stream_inference_to_window(
    video_path: Path,
    mediapipe_model_path: Path,
    pipeline,
    label_encoder,
    feat_config: dict,
    feature_columns: list[str],
    yaml_feedback: dict,
    pose_name: str,
    window_name: str,
    frame_step: int = 1,
    waitkey_ms: int = 1,
    display_width: int | None = None,
) -> dict:
    """
    Stream a video through the full inference pipeline and display it live
    in a cv2 popup window. Nothing is buffered - each frame is processed
    and shown immediately.

    Keyboard controls inside the window:
        Q / Esc   quit
        Space     pause / resume

    Args:
        video_path           : path to the input video
        mediapipe_model_path : path to the .task model file
        pipeline             : fitted sklearn Pipeline
        label_encoder        : fitted LabelEncoder
        feat_config          : engineered_features block from the pose YAML
        feature_columns      : feature names in training order
        yaml_feedback        : feedback strings from the pose YAML
        pose_name            : display name of the pose
        window_name          : cv2 window title
        frame_step           : process every Nth frame
        waitkey_ms           : cv2.waitKey delay; 1 = max speed, 33 = ~30 fps cap
        display_width        : resize window to this width (None = original resolution)

    Returns:
        Summary dict with processed frame count and per-class prediction counts.
    """
    options = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(
            model_asset_path=str(mediapipe_model_path)
        ),
        running_mode=RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
    )

    cap = cv2.VideoCapture(str(video_path))
    assert cap.isOpened(), f"Cannot open video: {video_path}"

    video_fps     = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video   : {video_path.name}")
    print(f"FPS     : {video_fps:.1f}  |  Total frames : {total_frames}")
    print(f"Window  : '{window_name}'  - press Q or Esc to quit, Space to pause")

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    if display_width is not None:
        # Set initial window size; user can still resize freely after
        orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        scale  = display_width / max(orig_w, 1)
        cv2.resizeWindow(window_name, display_width, int(orig_h * scale))

    frame_count     = 0
    processed_count = 0
    no_pose_count   = 0
    pred_counts: dict[str, int] = {}
    paused = False
    smoother = PredictionSmoother(SMOOTHING_WINDOW, MIN_HOLD_FRAMES)

    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            # Handle pause: keep polling waitKey without reading new frames
            if paused:
                key = cv2.waitKey(50) & 0xFF
                if key == ord(" "):
                    paused = False
                elif key in (ord("q"), 27):  # Q or Esc
                    print("Quit by user.")
                    break
                continue

            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            if (frame_count - 1) % frame_step != 0:
                continue

            processed_count += 1
            annotated    = frame.copy()
            timestamp_ms = int((frame_count / video_fps) * 1000)

            img_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            result   = landmarker.detect_for_video(mp_image, timestamp_ms)

            if not result.pose_landmarks:
                no_pose_count += 1
                cv2.putText(
                    annotated, "No pose detected",
                    (14, 50), cv2.FONT_HERSHEY_SIMPLEX,
                    1.0, (0, 60, 220), 2, cv2.LINE_AA,
                )
            else:
                landmarks = result.pose_landmarks[0]

                # 1. Skeleton
                draw_skeleton(annotated, landmarks)

                # 2. Feature engineering
                feature_vec = extract_features_from_landmarks(
                    landmarks, feat_config, feature_columns
                )

                if feature_vec is None:
                    cv2.putText(
                        annotated, "Degenerate pose",
                        (14, 50), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0, 100, 220), 2, cv2.LINE_AA,
                    )
                else:
                    # 3. Inference
                    X_frame    = feature_vec.reshape(1, -1)
                    pred_enc   = pipeline.predict(X_frame)[0]
                    pred_class = label_encoder.inverse_transform([pred_enc])[0]

                    probability = None
                    if HAS_PROBA:
                        proba_vec   = pipeline.predict_proba(X_frame)[0]
                        probability = float(proba_vec[pred_enc])


                    # Apply probability threshold first
                    if probability is not None and probability < PROBABILITY_THRESHOLD:
                        raw_class = "unknown"
                    else:
                        raw_class = pred_class

                    # Smooth over the sliding window
                    display_class    = smoother.update(raw_class)
                    display_feedback = yaml_feedback.get(display_class, "Pose unclear — please adjust your position.")

                    pred_counts[display_class] = pred_counts.get(display_class, 0) + 1

                    draw_prediction_overlay(
                        annotated,
                        pose_name=pose_name,
                        predicted_class=display_class,
                        probability=probability,
                        feedback=display_feedback,
                        frame_idx=frame_count,
                    )

            cv2.imshow(window_name, annotated)

            key = cv2.waitKey(waitkey_ms) & 0xFF
            if key in (ord("q"), 27):   # Q or Esc
                print("Quit by user.")
                break
            if key == ord(" "):         # Space
                paused = True

    cap.release()
    cv2.destroyWindow(window_name)

    summary = {
        "processed_frames" : processed_count,
        "no_pose_frames"   : no_pose_count,
        "pred_counts"      : pred_counts,
    }

    inferrable = max(1, processed_count - no_pose_count)
    print(f"\nDone. Processed {processed_count} frames | No-pose: {no_pose_count}")
    print("Prediction distribution:")
    for cls, cnt in sorted(pred_counts.items(), key=lambda x: -x[1]):
        print(f"  {cls:<25} {cnt:>5} frames  ({cnt / inferrable * 100:.1f}%)")

    return summary


summary = stream_inference_to_window(
    video_path           = VIDEO_PATH,
    mediapipe_model_path = MEDIAPIPE_MODEL_PATH,
    pipeline             = pipeline,
    label_encoder        = label_encoder,
    feat_config          = feat_config,
    feature_columns      = FEATURE_COLUMNS,
    yaml_feedback        = yaml_feedback,
    pose_name            = POSE_NAME,
    window_name          = WINDOW_NAME,
    frame_step           = FRAME_STEP,
    waitkey_ms           = WAITKEY_MS,
    display_width        = DISPLAY_WIDTH,
)

Video   : mountain_test.mp4
FPS     : 30.0  |  Total frames : 1800
Window  : 'mountain_pose - Pose Detection'  - press Q or Esc to quit, Space to pause

Done. Processed 1800 frames | No-pose: 0
Prediction distribution:
  correct                    1013 frames  (56.3%)
  wide_stance                 257 frames  (14.3%)
  tilted_head                 256 frames  (14.2%)
  arms_not_dropped            185 frames  (10.3%)
  unknown                      89 frames  (4.9%)


---
## Cell 9 - Re-run on a different video (optional)

Reuse the loaded pipeline without reloading anything - just point at a new video.

In [10]:
# Change this path and run this cell to test another video without reloading artefacts
ANOTHER_VIDEO = Path("test_videos/mountain_correct.mp4")

if ANOTHER_VIDEO.exists():
    stream_inference_to_window(
        video_path           = ANOTHER_VIDEO,
        mediapipe_model_path = MEDIAPIPE_MODEL_PATH,
        pipeline             = pipeline,
        label_encoder        = label_encoder,
        feat_config          = feat_config,
        feature_columns      = FEATURE_COLUMNS,
        yaml_feedback        = yaml_feedback,
        pose_name            = POSE_NAME,
        window_name          = WINDOW_NAME,
        frame_step           = FRAME_STEP,
        waitkey_ms           = WAITKEY_MS,
        display_width        = DISPLAY_WIDTH,
    )
else:
    print(f"File not found: {ANOTHER_VIDEO} - update the path above.")

Video   : mountain_correct.mp4
FPS     : 30.0  |  Total frames : 1800
Window  : 'mountain_pose - Pose Detection'  - press Q or Esc to quit, Space to pause
Quit by user.

Done. Processed 190 frames | No-pose: 0
Prediction distribution:
  correct                     184 frames  (96.8%)
  wide_stance                   6 frames  (3.2%)


---
## Cell 10 - Per-frame prediction summary (optional)

Tabular view of every processed frame - useful for spotting transitions or
debugging individual frames.

In [11]:
def build_prediction_log(
    video_path: Path,
    mediapipe_model_path: Path,
    pipeline,
    label_encoder,
    feat_config: dict,
    feature_columns: list[str],
    frame_step: int = 1,
) -> pd.DataFrame:
    """
    Replay the video and collect per-frame predictions into a DataFrame.
    Cheaper to run separately (no drawing overhead) for analysis purposes.

    Returns a DataFrame with columns:
        frame_number, predicted_class, probability (or NaN), feature values...
    """
    options = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(
            model_asset_path=str(mediapipe_model_path)
        ),
        running_mode=RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
    )

    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    rows  = []
    frame_count = 0

    with mp_vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            if (frame_count - 1) % frame_step != 0:
                continue

            timestamp_ms = int((frame_count / fps) * 1000)
            img_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
            result   = landmarker.detect_for_video(mp_image, timestamp_ms)

            row = {"frame_number": frame_count}

            if not result.pose_landmarks:
                row["predicted_class"] = "no_pose"
                row["probability"]     = float("nan")
                rows.append(row)
                continue

            landmarks   = result.pose_landmarks[0]
            feature_vec = extract_features_from_landmarks(
                landmarks, feat_config, feature_columns
            )

            if feature_vec is None:
                row["predicted_class"] = "degenerate_pose"
                row["probability"]     = float("nan")
                rows.append(row)
                continue

            for col, val in zip(feature_columns, feature_vec):
                row[col] = round(val, 6)

            X_frame   = feature_vec.reshape(1, -1)
            pred_enc  = pipeline.predict(X_frame)[0]
            pred_class = label_encoder.inverse_transform([pred_enc])[0]
            row["predicted_class"] = pred_class

            if HAS_PROBA:
                proba_vec = pipeline.predict_proba(X_frame)[0]
                row["probability"] = round(float(proba_vec[pred_enc]), 4)
            else:
                row["probability"] = float("nan")

            rows.append(row)

    cap.release()
    return pd.DataFrame(rows)


log_df = build_prediction_log(
    video_path           = VIDEO_PATH,
    mediapipe_model_path = MEDIAPIPE_MODEL_PATH,
    pipeline             = pipeline,
    label_encoder        = label_encoder,
    feat_config          = feat_config,
    feature_columns      = FEATURE_COLUMNS,
    frame_step           = FRAME_STEP,
)

print(f"Per-frame log: {len(log_df)} rows")
print("\nClass distribution across frames:")
print(log_df["predicted_class"].value_counts().to_string())
print()
log_df.head(20)

Per-frame log: 1800 rows

Class distribution across frames:
predicted_class
correct             1649
wide_stance           84
arms_not_dropped      64
tilted_head            3



,frame_number,left_torso_hip_angle,right_torso_hip_angle,left_shoulder_arm_angle,right_shoulder_arm_angle,neck_tilt_angle,feet_distance_normalized,ear_shoulder_lateral_delta,plumb_line_alignment,predicted_class,probability
0,1,145.767378,146.890830,52.853389,52.468322,161.519439,0.501749,4.926520,0.343908,correct,0.9200
1,2,144.841434,144.441445,51.020391,53.472096,161.584053,0.525211,5.405675,0.335483,correct,0.9351
2,3,145.515061,143.738112,51.097866,53.148175,161.484060,0.548517,4.694457,0.352837,correct,0.9260
3,4,142.029745,143.355949,50.998575,53.132307,161.629194,0.579030,4.811424,0.344008,correct,0.9319
4,5,143.881192,144.458065,50.584582,51.324748,161.284299,0.572698,4.559284,0.357580,correct,0.9186
5,6,143.474514,144.370685,50.934445,51.886321,161.292535,0.556831,4.567906,0.355656,correct,0.9296
6,7,149.938498,146.749724,51.021998,51.752153,160.098810,0.552413,4.220739,0.363053,correct,0.8994
7,8,145.285189,144.501309,49.958869,52.726104,161.638573,0.549534,4.265847,0.329789,correct,0.9320
8,9,144.965164,144.239562,50.112116,52.709691,161.735268,0.550391,4.077838,0.333082,correct,0.9314
9,10,142.868483,143.274094,50.660675,53.298461,162.394668,0.544849,3.982703,0.325106,correct,0.9391
